#### Models can request to call tools that perform tasks such as fetching data fromm database , searching the web or running code. Tools are pairing of:
1. A schema inckuding the name of tool, a decription and or argument definitions (often a JSON schema)
2. a function or subroutime to excuete

In [1]:
## integrate with Groq
import os
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq # another way of using groq

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("why do parrots talk ?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. I need to explain the reasons behind their ability to mimic human speech. Let me start by recalling what I know about parrots. They are part of the Psittaciformes order, right? And they have a syrinx for vocalization. But why do they mimic sounds?\n\nFirst, maybe it\'s related to their social behavior. Parrots are social animals, so in the wild, they learn to communicate with each other. Mimicking sounds could be a way to bond or integrate into a group. If they\'re kept as pets, they might mimic their owners to fit into their human social group. That makes sense.\n\nThen there\'s the aspect of learning through repetition. Parrots have a good memory, so they can remember and repeat sounds they hear. They might associate certain words with specific outcomes, like getting food or attention. Positive reinforcement could encourage them to talk more. For example, if a parrot says a word and gets a treat, it\'s more lik

In [2]:
from langchain.tools import tool
## use tool as decorator to mark that function as tool
@tool
def get_weather(location:str)->str:
    """Get weather at a location"""
    return f"its sunny in {location}"
model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("what the weather like in Dallas, TX")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by model
    print(f"Tool:{tool_call['name']}")
    print(f"Tool:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Dallas, TX. I need to use the get_weather function. The function requires a location parameter. The user provided "Dallas, TX" which is a valid location. I should structure the tool call with the name as get_weather and the arguments including location as Dallas, TX. Let me make sure there are no typos. Everything looks good. Time to format the JSON response within the tool_call tags.\n', 'tool_calls': [{'id': 'rg6q56bse', 'function': {'arguments': '{"location":"Dallas, TX"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 153, 'total_tokens': 271, 'completion_time': 0.216777594, 'completion_tokens_details': {'reasoning_tokens': 92}, 'prompt_time': 0.008932987, 'prompt_tokens_details': None, 'queue_time': 1.858539368, 'total_time': 0.225710581}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36'

In [ ]:
#### Tool Execution Loops
# Step 1 : Model generation tool calls
messages = [{"role":"user","content":"what is the weather in Houston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2 : Execure tools and collect results

for tool_call in ai_msg.tool_calls:
    # Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call) 
    messages.append(tool_result)

# Step 3 :  Pass the results back to the model for final response

final_response= model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Houston is 72 F and sunny"

The weather in Houston is sunny right now. If you're there, it's a great day to enjoy outdoor activities! ☀️


In [7]:
messages

[{'role': 'user', 'content': 'what is the weather in Houston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Houston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Houston is the location they mentioned. I need to call that function with "Houston" as the argument. I\'ll make sure the JSON is correctly formatted with the location parameter. No other parameters are needed since the function only requires location. Let me double-check the syntax to avoid any errors.\n', 'tool_calls': [{'id': 'vbxhrydr8', 'function': {'arguments': '{"location":"Houston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 152, 'total_tokens': 266, 'completion_time': 0.184159278, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.007376965, 'prompt_tokens_details': None, 'queue_time': 0.0229